In [ ]:
!pip install -q scikit-learn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch

folk_path = "/content/drive/My Drive/code/folkdataset.pt"
jazz_path = "/content/drive/My Drive/code/jazzdataset.pt"

folk_data = torch.load(folk_path, map_location="cpu")
jazz_data = torch.load(jazz_path, map_location="cpu")

print("folk_data keys:", folk_data.keys())
print("jazz_data keys:", jazz_data.keys())

folk_mel = folk_data["folk_feature"]
jazz_mel = jazz_data["jazz_feature"]

print("folk_mel shape:", folk_mel.shape)
print("jazz_mel shape:", jazz_mel.shape)


folk_data keys: dict_keys(['folk_info', 'folk_feature', 'folk_label'])
jazz_data keys: dict_keys(['jazz_info', 'jazz_feature', 'jazz_label'])
folk_mel shape: torch.Size([2207, 80, 616])
jazz_mel shape: torch.Size([2066, 80, 616])


In [ ]:
import numpy as np

def flatten_mel(mel):
    # mel: (N, 80, 616)
    N = mel.shape[0]
    return mel.reshape(N, -1)

folk_flat = flatten_mel(folk_mel)  # (2207, 49280)
jazz_flat = flatten_mel(jazz_mel)  # (2066, 49280)


In [ ]:
from sklearn.preprocessing import StandardScaler

all_flat = np.vstack([folk_flat, jazz_flat])
scaler = StandardScaler()
all_scaled = scaler.fit_transform(all_flat)

folk_scaled = scaler.transform(folk_flat)
jazz_scaled = scaler.transform(jazz_flat)


Baseline method:

In [ ]:
import numpy as np

folk_mean = folk_scaled.mean(axis=0)
jazz_mean = jazz_scaled.mean(axis=0)
style_vector = jazz_mean - folk_mean

folk_styled_scaled = folk_scaled + style_vector

folk_styled_flat = scaler.inverse_transform(folk_styled_scaled)

N_folk, F, T = folk_mel.shape
folk_styled_mel = folk_styled_flat.reshape(N_folk, F, T)

print(folk_styled_mel.shape)


(2207, 80, 616)


In [ ]:
import numpy as np

# folk_mel: (N_folk, 80, 616)
# jazz_mel: (N_jazz, 80, 616)

N_folk, F, T = folk_mel.shape
N_jazz, _, _ = jazz_mel.shape

folk_flat = folk_mel.reshape(N_folk, -1)   # (N_folk, F*T)
jazz_flat = jazz_mel.reshape(N_jazz, -1)   # (N_jazz, F*T)

folk_mean = folk_flat.mean(axis=0)         # (F*T,)
jazz_mean = jazz_flat.mean(axis=0)         # (F*T,)
style_vec = jazz_mean - folk_mean

folk_styled_flat = folk_flat + style_vec   # (N_folk, F*T)

folk_styled_mel = folk_styled_flat.reshape(N_folk, F, T)
print(folk_styled_mel.shape)  # (N_folk, 80, 616)


torch.Size([2207, 80, 616])


In [ ]:
import torch

torch.save({
    "folk_jazz_Baseline": torch.tensor(folk_styled_mel).float(),
}, "folk_jazz_Baseline.pt")

print("✔ Baseline saved to folk_jazz_Baseline.pt")



/tmp/ipython-input-83243197.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "folk_jazz_Baseline": torch.tensor(folk_styled_mel).float(),


✔ Baseline saved to folk_jazz_Baseline.pt


PCA:

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
all_pca = pca.fit_transform(all_scaled)

folk_pca = all_pca[:len(folk_flat)]
jazz_pca = all_pca[len(folk_flat):]


In [ ]:
n_content = 2   # PC1,PC2


In [ ]:
n_content = 2
folk_content = folk_pca[:, :n_content]
folk_style   = folk_pca[:, n_content:]

jazz_content = jazz_pca[:, :n_content]
jazz_style   = jazz_pca[:, n_content:]


In [ ]:
jazz_style_mean = jazz_style.mean(axis=0, keepdims=True)  # (1, 49278)


In [ ]:
folk_styled_pca = np.hstack([
    folk_content,
    np.repeat(jazz_style_mean, len(folk_content), axis=0)
])


In [ ]:
folk_styled_scaled = pca.inverse_transform(folk_styled_pca)
folk_styled_flat = scaler.inverse_transform(folk_styled_scaled)


In [ ]:
folk_styled_mel = folk_styled_flat.reshape(-1, 80, 616)


In [ ]:
torch.save({
    "folk_jazz_PCA": torch.from_numpy(folk_styled_mel).float()
}, "folk_jazz_PCA.pt")



KNN：

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=5)
knn.fit(jazz_scaled)

NearestNeighbors()

In [ ]:
styled_scaled = scaler.transform(folk_styled_flat)
distances, indices = knn.kneighbors(styled_scaled)  # (2207,5)

neighbors = jazz_flat[indices]   # (2207,5,49280)
folk_refined_flat = neighbors.mean(axis=1)  # (2207,49280)
folk_refined_mel = folk_refined_flat.reshape(-1, 80, 616)


In [ ]:
folk_refined_mel



tensor([[[-41.1366, -31.8118, -32.3964,  ..., -21.9226, -21.9001, -22.4857],
         [-34.9250, -20.4594, -16.5607,  ..., -14.9233, -14.0013, -13.1007],
         [-32.3193, -17.2610, -12.2495,  ..., -17.8524, -18.3277, -17.2632],
         ...,
         [-70.3594, -53.6423, -48.8843,  ..., -42.7588, -43.7066, -45.1107],
         [-72.0420, -55.6229, -50.7321,  ..., -43.9855, -45.0497, -46.4581],
         [-75.1892, -62.4805, -57.4547,  ..., -49.8436, -51.1353, -53.6907]],

        [[-45.0942, -28.8549, -25.9757,  ..., -17.6110, -18.0292, -18.0241],
         [-39.9707, -22.3587, -17.9290,  ..., -16.6785, -13.0671, -11.6414],
         [-35.4944, -20.2801, -16.0375,  ..., -14.5090,  -9.3925, -10.1515],
         ...,
         [-63.9981, -52.1748, -50.3354,  ..., -51.4432, -51.9980, -53.2645],
         [-65.8568, -52.8083, -49.3930,  ..., -52.3332, -52.2931, -53.2644],
         [-68.7185, -57.8872, -54.4870,  ..., -56.1952, -56.3165, -56.7750]],

        [[-38.5630, -31.4537, -32.6802,  ...

In [ ]:
torch.save({
    "folk_jazz_KNN": folk_refined_mel.detach().cpu().clone()
}, "folk_jazz_style.pt")
